In [22]:
import os
from random import random

print(os.getcwd())  # Выведет текущую рабочую директорию

/Users/smetdenis/Work/smetdenis/ml-notes/grokking-dl


In [21]:
import pandas as pd
from srsly.msgpack import epoch

df = pd.read_csv('./IMDB_Dataset.csv')

In [3]:
from nltk.tokenize import TweetTokenizer


def get_tokens(text):
    tokenizer = TweetTokenizer()
    return tokenizer.tokenize(text.lower())


tokens = list(map(lambda x: set(get_tokens(x)), df.review))

In [5]:
vocab = set()
for sent in tokens:
    for word in sent:
        if (len(word) > 0):
            vocab.add(word)
vocab = list(vocab)
len(vocab)

150049

In [6]:
word2index = {}
for i, word in enumerate(vocab):
    word2index[word] = i

In [7]:
target_dataset = list()
for label in df.sentiment:
    if label == 'positive':
        target_dataset.append(1)
    else:
        target_dataset.append(0)

In [8]:
input_dataset = list()
for sent in tokens:
    sent_inx = list()
    for word in sent:
        try:
            sent_inx.append(word2index[word])
        except:
            ""
    input_dataset.append(list(set(sent_inx)))

In [9]:
import numpy as np

np.random.seed(1)


def sigmoid(x):
    return 1 / (1 + np.exp(-x))


alpha = 0.01
epochs = 2
hidden_size = 100

weight_0_1 = 0.2 * np.random.random((len(vocab), hidden_size)) - 0.1
weight_1_2 = 0.2 * np.random.random((hidden_size, 1)) - 0.1

for epoch in range(epochs):
    correct, total = (0, 0)
    for i in range(len(input_dataset) - 1000):
        x_tran, y_train = input_dataset[i], target_dataset[i]
        layer_1 = sigmoid(np.sum(weight_0_1[x_tran], axis=0))
        layer_2 = sigmoid(layer_1.dot(weight_1_2))

        layer_2_delta = layer_2 - y_train
        layer_1_delta = layer_2_delta.dot(weight_1_2.T)

        weight_0_1[x_tran] -= layer_1_delta * alpha
        weight_1_2 -= np.outer(layer_1, layer_2_delta) * alpha

        if (np.abs(layer_2_delta) < 0.5):
            correct += 1
        total += 1

        if (i % 1000 == 0):
            progress = str(round(i / float(len(input_dataset)) * 100, 2))
            print("Progress:" + progress +
                  " Epoch:" + str(epoch) +
                  " Train Acc:" + str(round(correct / float(total) * 100, 2)))

    # test set evaluation
    correct_error, total_error = (0, 0)
    for i in range(len(input_dataset) - 1000, len(input_dataset)):
        x, y = input_dataset[i], target_dataset[i]
        layer_1 = sigmoid(np.sum(weight_0_1[x], axis=0))
        layer_2 = sigmoid(layer_1 @ weight_1_2)
        if (np.abs(layer_2 - y) < 0.5):
            correct_error += 1
        total_error += 1
    print(f"Test accuracy: {correct_error / float(total_error)}")


Progress:0.0 Epoch:0 Train Acc:100.0
Progress:2.0 Epoch:0 Train Acc:57.94
Progress:4.0 Epoch:0 Train Acc:65.37
Progress:6.0 Epoch:0 Train Acc:69.51
Progress:8.0 Epoch:0 Train Acc:72.68
Progress:10.0 Epoch:0 Train Acc:74.87
Progress:12.0 Epoch:0 Train Acc:76.39
Progress:14.0 Epoch:0 Train Acc:77.65
Progress:16.0 Epoch:0 Train Acc:78.43
Progress:18.0 Epoch:0 Train Acc:79.14
Progress:20.0 Epoch:0 Train Acc:79.73
Progress:22.0 Epoch:0 Train Acc:80.39
Progress:24.0 Epoch:0 Train Acc:80.86
Progress:26.0 Epoch:0 Train Acc:81.3
Progress:28.0 Epoch:0 Train Acc:81.47
Progress:30.0 Epoch:0 Train Acc:81.85
Progress:32.0 Epoch:0 Train Acc:82.19
Progress:34.0 Epoch:0 Train Acc:82.43
Progress:36.0 Epoch:0 Train Acc:82.62
Progress:38.0 Epoch:0 Train Acc:82.77
Progress:40.0 Epoch:0 Train Acc:83.06
Progress:42.0 Epoch:0 Train Acc:83.18
Progress:44.0 Epoch:0 Train Acc:83.29
Progress:46.0 Epoch:0 Train Acc:83.48
Progress:48.0 Epoch:0 Train Acc:83.53
Progress:50.0 Epoch:0 Train Acc:83.68
Progress:52.0 Epoc

In [16]:
from collections import Counter
import math


def similar(target_word):
    target_word = word2index[target_word]
    scores = Counter()

    for word, word_index in word2index.items():
        raw_diff = weight_0_1[word_index] - (weight_0_1[target_word])
        square_diff = raw_diff ** 2
        scores[word] = - math.sqrt(sum(square_diff))

    return scores.most_common(5)

In [20]:
similar('terrible'), similar('dull')

([('terrible', -0.0),
  ('poorly', -0.8186289704833116),
  ('disappointment', -0.8307340815121423),
  ('fails', -0.8538081454616075),
  ('boring', -0.8837494489442794)],
 [('dull', -0.0),
  ('boring', -0.7467923866855882),
  ('fails', -0.8248140969921449),
  ('poor', -0.8618127784419302),
  ('poorly', -0.8713886537260019)])